# Segmentação e regiões conectadas

Limiarização, bordas, refinamento de máscaras e rotulagem de regiões para imagens grayscale.

**Pré-requisitos:** histogramas, filtros espaciais e morfologia binária.

## Convenções

A API recebe imagens grayscale 2D `uint8`. As máscaras usam `0` para fundo e `255` para primeiro plano. Imagens coloridas devem ser convertidas para grayscale antes do uso. Em componentes conectados, o fundo tem rótulo `0`; as regiões recebem IDs consecutivos a partir de `1`, em ordem de varredura por linhas.

## 1. Instalação

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit.modules.image_segmenter import ImageSegmenter
from dip_toolkit.modules.morphology import MorphologyProcessor

segmenter = ImageSegmenter()
morphology = MorphologyProcessor()

## 2. Limiar global e Otsu

No limiar global, `bright` seleciona intensidades maiores ou iguais ao valor escolhido; `dark` seleciona as menores ou iguais. Otsu determina esse limiar automaticamente quando há duas classes de intensidade.

In [ ]:
image = np.full((80, 160), 40, dtype=np.uint8)
image[20:60, 45:115] = 180
global_mask = segmenter.global_threshold(image, 100)
otsu_result = segmenter.otsu(image)

figure, axes = plt.subplots(1, 3, figsize=(12, 3))
for axis, data, title in zip(
    axes,
    (image, global_mask, otsu_result.mask),
    ("Imagem", "Limiar global", f"Otsu = {otsu_result.threshold}"),
    strict=True,
):
    axis.imshow(data, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
plt.show()

## 3. Limiar adaptativo

Quando a iluminação varia na imagem, o limiar adaptativo usa a média local de uma janela ímpar `block_size`, menos a constante `constant`.

In [ ]:
columns = np.linspace(30, 190, 180, dtype=np.uint8)
uneven = np.tile(columns, (100, 1))
uneven[30:70, 25:55] = 230
uneven[30:70, 120:150] = 230
adaptive_mask = segmenter.adaptive_threshold(uneven, block_size=21, constant=8)

figure, axes = plt.subplots(1, 2, figsize=(10, 3))
for axis, data, title in zip(
    axes,
    (uneven, adaptive_mask),
    ("Iluminação não uniforme", "Limiar adaptativo"),
    strict=True,
):
    axis.imshow(data, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
plt.show()

## 4. Bordas e refinamento

A segmentação por bordas reutiliza Canny com limiares explícitos. A abertura remove ruídos pequenos e o fechamento preenche lacunas pequenas, reutilizando a morfologia da DIP-09.

In [ ]:
edges = segmenter.edge_segmentation(image, 50, 150)
noisy = global_mask.copy()
noisy[5, 5] = 255
noisy[40, 80] = 0
element = morphology.create_structuring_element("rectangle", (3, 3))
refined = segmenter.refine_mask(noisy, element, ["opening", "closing"])

figure, axes = plt.subplots(1, 3, figsize=(12, 3))
for axis, data, title in zip(
    axes,
    (edges, noisy, refined),
    ("Bordas por Canny", "Máscara com ruído", "Máscara refinada"),
    strict=True,
):
    axis.imshow(data, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
plt.show()

## 5. Componentes conectados

A conectividade 4 considera vizinhos horizontal e vertical; a conectividade 8 também considera diagonais.

In [ ]:
regions = segmenter.connected_components(refined, connectivity=8)
diagonal = np.array([[255, 0], [0, 255]], dtype=np.uint8)
print(f"Regiões da máscara refinada: {regions.region_ids}")
print(segmenter.connected_components(diagonal, connectivity=4).region_ids)
print(segmenter.connected_components(diagonal, connectivity=8).region_ids)

figure, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(regions.mask, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Máscara")
axes[1].imshow(regions.labels, cmap="nipy_spectral")
axes[1].set_title("Mapa de rótulos")
for axis in axes:
    axis.axis("off")
plt.show()

## Experimente

Altere as polaridades, os limiares e o elemento estruturante. Crie pixels que se tocam somente na diagonal para comparar conectividades 4 e 8.